# 01 — Grid Definition (Paris / EUBUCCO)

Defines a regular 150m grid over Paris and derives the **Y variable** (`zone_type`) from EUBUCCO building subtypes.

**Replaces:** NYC PLUTO with EUBUCCO open building dataset.

**Method:**
1. Stream EUBUCCO buildings for Paris (NUTS2: FR10) from S3
2. Filter to residential, commercial, industrial subtypes
3. Generate 150m x 150m regular grid covering Paris bounding box
4. Clip grid to convex hull of buildings (removes water/empty areas)
5. Assign buildings to grid cells
6. Derive zone_type by plurality of building subtype per cell

**Output columns:** `cell_id`, `cell_lat`, `cell_lon`, `zone_type`, `cell_building_count`

**Output file:** `csv/Paris/01_grid_definition.csv`

In [1]:
# ── Config ────────────────────────────────────────────
PARIS_CONFIG = "paris.json"

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import os
import math
from scipy.spatial import ConvexHull
from shapely.geometry import Polygon, Point

with open(PARIS_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

NUTS_CODE     = config["nuts_code"]
CELL_SIZE_M   = config["grid_cell_size_m"]
MIN_BUILDINGS = config["min_buildings_per_cell"]
CSV_DIR       = config["csv_dir"]
TARGET_LABELS = config["target_labels"]

os.makedirs(CSV_DIR, exist_ok=True)

print(f"City:          {config['city']}")
print(f"NUTS code:     {NUTS_CODE}")
print(f"Grid cell:     {CELL_SIZE_M}m x {CELL_SIZE_M}m")
print(f"Min buildings: {MIN_BUILDINGS}")
print(f"Labels:        {TARGET_LABELS}")

City:          Paris, France
NUTS code:     FR10
Grid cell:     150m x 150m
Min buildings: 3
Labels:        ['residential', 'commercial', 'industrial']


In [3]:
# ── Stream EUBUCCO buildings from S3 ─────────────────
storage_opts = {
    "anon": True,
    "client_kwargs": {"endpoint_url": "https://s3.eubucco.com"}
}

path = f"s3://eubucco/v0.2/buildings/parquet/nuts_id={NUTS_CODE}/{NUTS_CODE}.parquet"
print(f"Streaming EUBUCCO from: {path}")

gdf = gpd.read_parquet(path, storage_options=storage_opts)
print(f"Loaded {len(gdf):,} buildings")

# Reproject to WGS84 and extract centroids
gdf = gdf.to_crs("EPSG:4326")
gdf["latitude"]  = gdf.geometry.centroid.y
gdf["longitude"] = gdf.geometry.centroid.x

print(f"\nSubtype distribution:")
print(gdf["subtype"].value_counts().to_string())

Streaming EUBUCCO from: s3://eubucco/v0.2/buildings/parquet/nuts_id=FR10/FR10.parquet


Loaded 3,594,995 buildings


C:\Users\Hani\AppData\Local\Temp\ipykernel_14780\2583268541.py:15: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["latitude"]  = gdf.geometry.centroid.y


C:\Users\Hani\AppData\Local\Temp\ipykernel_14780\2583268541.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["longitude"] = gdf.geometry.centroid.x



Subtype distribution:
subtype
detached         2264659
apartment         439545
others            416612
commercial        228183
industrial        118456
agricultural       71440
semi-detached      30589
public             14282
terraced           11229


In [4]:
# ── Filter and label buildings ────────────────────────

# residential = top-level 'type' field (covers apartment, detached, etc.)
# commercial and industrial = 'subtype' field
conditions = (
    (gdf["type"] == "residential") |
    (gdf["subtype"] == "commercial") |
    (gdf["subtype"] == "industrial")
)
gdf_f = gdf[conditions].copy()
gdf_f["zone_cat"] = gdf_f.apply(
    lambda r: "residential" if r["type"] == "residential" else r["subtype"],
    axis=1
)
gdf_f = gdf_f.dropna(subset=["latitude", "longitude"]).copy()

print(f"Buildings after filter: {len(gdf_f):,}")
print(gdf_f["zone_cat"].value_counts().to_string())

Buildings after filter: 3,092,661
zone_cat
residential    2746022
commercial      228183
industrial      118456


In [5]:
# ── Generate regular grid ─────────────────────────────
REF_LAT  = gdf_f["latitude"].mean()
LAT_STEP = CELL_SIZE_M / 111_000
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))

print(f"Reference latitude: {REF_LAT:.4f}")
print(f"Grid steps: lat={LAT_STEP:.6f}, lon={LON_STEP:.6f}")

BUFFER  = LAT_STEP
LAT_MIN = gdf_f["latitude"].min()  - BUFFER
LAT_MAX = gdf_f["latitude"].max()  + BUFFER
LON_MIN = gdf_f["longitude"].min() - BUFFER
LON_MAX = gdf_f["longitude"].max() + BUFFER

n_rows = int(math.ceil((LAT_MAX - LAT_MIN) / LAT_STEP))
n_cols = int(math.ceil((LON_MAX - LON_MIN) / LON_STEP))
print(f"Grid: {n_rows} rows x {n_cols} cols = {n_rows * n_cols:,} total cells")

# Convex hull to clip water/empty cells
coords = gdf_f[["longitude", "latitude"]].values
hull = ConvexHull(coords)
hull_polygon = Polygon(coords[hull.vertices])
print(f"Convex hull computed.")

Reference latitude: 48.8089
Grid steps: lat=0.001351, lon=0.002052
Grid: 829 rows x 1022 cols = 847,238 total cells


Convex hull computed.


In [6]:
# ── Assign buildings to grid cells ───────────────────
gdf_f["grid_row"] = ((gdf_f["latitude"]  - LAT_MIN) / LAT_STEP).astype(int)
gdf_f["grid_col"] = ((gdf_f["longitude"] - LON_MIN) / LON_STEP).astype(int)
gdf_f["cell_id"]  = "r" + gdf_f["grid_row"].astype(str).str.zfill(4) + "_c" + gdf_f["grid_col"].astype(str).str.zfill(4)

PLURALITY_THRESHOLD = 0.40

cell_records = []
skipped_hull = 0
skipped_min  = 0

for (row, col), group in gdf_f.groupby(["grid_row", "grid_col"]):
    cell_lat = LAT_MIN + (row + 0.5) * LAT_STEP
    cell_lon = LON_MIN + (col + 0.5) * LON_STEP

    if not hull_polygon.contains(Point(cell_lon, cell_lat)):
        skipped_hull += 1
        continue

    if len(group) < MIN_BUILDINGS:
        skipped_min += 1
        continue

    # Plurality zone type
    counts = group["zone_cat"].value_counts()
    dominant       = counts.idxmax()
    dominant_ratio = counts[dominant] / len(group)
    zone_type = dominant if dominant_ratio >= PLURALITY_THRESHOLD else "mixed"

    cell_records.append({
        "cell_id":              f"r{row:04d}_c{col:04d}",
        "cell_lat":             round(cell_lat, 7),
        "cell_lon":             round(cell_lon, 7),
        "zone_type":            zone_type,
        "cell_building_count":  len(group),
    })

df_grid = pd.DataFrame(cell_records)
print(f"Skipped (outside hull): {skipped_hull}")
print(f"Skipped (< {MIN_BUILDINGS} buildings): {skipped_min}")
print(f"Final grid cells: {len(df_grid):,}")
print(df_grid["zone_type"].value_counts().to_string())

Skipped (outside hull): 16
Skipped (< 3 buildings): 35192
Final grid cells: 120,331
zone_type
residential    101989
industrial      10458
commercial       7458
mixed             426


In [7]:
# ── Save grid parameters to config ───────────────────
# So other notebooks can reconstruct the same grid
config["grid_params"] = {
    "lat_min": LAT_MIN, "lon_min": LON_MIN,
    "lat_step": LAT_STEP, "lon_step": LON_STEP,
    "ref_lat": REF_LAT
}
with open(PARIS_CONFIG, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)
print("Grid params saved to paris.json")

# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/01_grid_definition.csv"
df_grid.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_grid)} rows x {df_grid.shape[1]} cols)")
df_grid.head(10)

Grid params saved to paris.json


Saved: csv/Paris/01_grid_definition.csv  (120331 rows x 5 cols)


,cell_id,cell_lat,cell_lon,zone_type,cell_building_count
0,r0004_c0596,48.125308,2.673187,residential,5
1,r0004_c0597,48.125308,2.675239,residential,3
2,r0004_c0606,48.125308,2.693706,residential,13
3,r0006_c0521,48.128010,2.519291,residential,8
4,r0006_c0522,48.128010,2.521343,residential,3
5,r0006_c0601,48.128010,2.683447,residential,4
6,r0007_c0521,48.129362,2.519291,residential,3
7,r0007_c0594,48.129362,2.669083,residential,7
8,r0007_c0595,48.129362,2.671135,residential,3
9,r0007_c0596,48.129362,2.673187,residential,5


In [8]:
# ── Summary ───────────────────────────────────────────
print("Zone type breakdown:")
for zt in sorted(df_grid["zone_type"].unique()):
    subset = df_grid[df_grid["zone_type"] == zt]
    print(f"  {zt:<20s} {len(subset):>5d} cells  "
          f"(avg {subset['cell_building_count'].mean():.0f} buildings/cell)")
print(f"\nTotal cells: {len(df_grid):,}")
print(f"Lat range: {df_grid['cell_lat'].min():.4f} — {df_grid['cell_lat'].max():.4f}")
print(f"Lon range: {df_grid['cell_lon'].min():.4f} — {df_grid['cell_lon'].max():.4f}")

Zone type breakdown:
  commercial            7458 cells  (avg 10 buildings/cell)
  industrial           10458 cells  (avg 7 buildings/cell)
  mixed                  426 cells  (avg 6 buildings/cell)
  residential          101989 cells  (avg 28 buildings/cell)

Total cells: 120,331
Lat range: 48.1253 — 49.2361
Lon range: 1.4523 — 3.5206
